In [0]:
dbutils.widgets.dropdown(name = "environment", defaultValue= "dev",choices= ["dev","prd","qa"],label = "select Environment")
env = dbutils.widgets.get("environment")
# print(env)


silverTablName = f"saleslake_{env}.silver_{env}.cleanedsales"
print("silverTablName:", silverTablName)
goldTablName = f"saleslake_{env}.gold_{env}.refinedsales"
print("goldTablName:",goldTablName )


In [0]:
spark.sql(f"""
MERGE INTO {goldTablName} tgt
USING (
    WITH latest_inv_silver AS (
        -- Step 1: Filter only new/updated silver records since last gold load
        SELECT *
        FROM {silverTablName}
        WHERE ingest_ts > (
            SELECT COALESCE(MAX(start_effective_ts), TO_TIMESTAMP('1990-01-01', 'yyyy-MM-dd'))
            FROM {goldTablName}
        )
    ),

    latest_rm_dup_silver AS (
        -- Step 2: Deduplicate — keep latest record per sale_id
        SELECT * FROM (
            SELECT *,
                   ROW_NUMBER() OVER (PARTITION BY sale_id ORDER BY ingest_ts DESC) AS rn
            FROM latest_inv_silver
        ) WHERE rn = 1
    ),

    silver_gold_rec AS (
        -- Step 3: Join with active gold records and classify each record
        SELECT
            s.*,
            g.sale_sk              AS sale_sk,
            g.is_active            AS is_active,
            g.rec_version          AS rec_version,
            g.start_effective_ts   AS start_effective_ts,
            g.end_effective_ts     AS end_effective_ts,
            CASE
                WHEN g.sale_sk IS NULL THEN 1
                ELSE g.rec_version + 1
            END AS new_rec_version,
            CASE
                WHEN g.sale_sk IS NULL THEN 'NEW'
                WHEN NOT (s.product_id      <=> g.product_id)
                  OR NOT (s.store_id        <=> g.store_id)
                  OR NOT (s.customer_id     <=> g.customer_id)
                  OR NOT (s.region_id       <=> g.region_id)
                  OR NOT (s.quantity        <=> g.quantity)
                  OR NOT (s.unit_price      <=> g.unit_price)
                  OR NOT (s.gross_amount    <=> g.gross_amount)
                  OR NOT (s.discount_code   <=> g.discount_code)
                  OR NOT (s.discount_amount <=> g.discount_amount)
                  OR NOT (s.net_amount      <=> g.net_amount)
                  OR NOT (s.cost_amount     <=> g.cost_amount)
                  OR NOT (s.sale_date       <=> g.sale_date)
                  OR NOT (s.payment_method  <=> g.payment_method)
                  OR NOT (s.channel         <=> g.channel)
                THEN 'CHANGE'
                ELSE 'NO_CHANGE'
            END AS rec_flag
        FROM latest_rm_dup_silver s
        LEFT JOIN (SELECT * FROM {goldTablName} WHERE is_active = 'Y') g
               ON s.sale_id = g.sale_id
    ),

    insert_flag AS (
        -- Step 4a: New version row to INSERT (for both NEW and CHANGE records)
        SELECT
            NULL    AS sale_merge_key,
            sale_id, product_id, store_id, customer_id, region_id,
            quantity, unit_price, gross_amount, discount_code,
            discount_amount, net_amount, cost_amount,
            sale_date, payment_method, channel,
            sale_sk, is_active, rec_version,
            start_effective_ts, end_effective_ts,
            new_rec_version, rec_flag,
            'INSERT' AS merge_flag
        FROM silver_gold_rec
        WHERE rec_flag IN ('NEW', 'CHANGE')
    ),

    changed_flag AS (
        -- Step 4b: Existing active row to CLOSE (UPDATE) for CHANGE records only
        SELECT
            sale_id AS sale_merge_key,
            sale_id, product_id, store_id, customer_id, region_id,
            quantity, unit_price, gross_amount, discount_code,
            discount_amount, net_amount, cost_amount,
            sale_date, payment_method, channel,
            sale_sk, is_active, rec_version,
            start_effective_ts, end_effective_ts,
            new_rec_version, rec_flag,
            'UPDATE' AS merge_flag
        FROM silver_gold_rec
        WHERE rec_flag = 'CHANGE'
    )

    -- Step 5: Combine both INSERT and UPDATE streams
    SELECT * FROM changed_flag
    UNION ALL
    SELECT * FROM insert_flag

) src
ON tgt.sale_id = src.sale_merge_key AND tgt.is_active = 'Y'

-- Close the old active record
WHEN MATCHED AND src.merge_flag = 'UPDATE' THEN UPDATE SET
    tgt.is_active        = 'N',
    tgt.end_effective_ts = CURRENT_TIMESTAMP()

-- Insert new record (new entry or new version of changed record)
WHEN NOT MATCHED AND src.merge_flag = 'INSERT' THEN INSERT (
    sale_id, product_id, store_id, customer_id, region_id,
    quantity, unit_price, gross_amount, discount_code,
    discount_amount, net_amount, cost_amount,
    sale_date, payment_method, channel,
    is_active, rec_version,
    start_effective_ts, end_effective_ts
)
VALUES (
    src.sale_id, src.product_id, src.store_id, src.customer_id, src.region_id,
    src.quantity, src.unit_price, src.gross_amount, src.discount_code,
    src.discount_amount, src.net_amount, src.cost_amount,
    src.sale_date, src.payment_method, src.channel,
    'Y', src.new_rec_version,
    CURRENT_TIMESTAMP(), TO_TIMESTAMP('9999-12-31', 'yyyy-MM-dd')
)
""")

In [0]:
%sql
SELECT count(*) FROM saleslake_dev.gold_dev.refinedsales;

--SELECT * FROM saleslake_dev.silver_dev.cleanedsales;
